# Task Switching v3

**Track:** Executive Functions
**Cognitive Ability:** Cognitive flexibility / task-set reconfiguration

## Methodology

Tests the ability to switch between different classification rules rapidly.

### v3 Changes (breaking ceiling effects):
- **4 compositional rules** requiring multi-step computation:
  - Rule A: Is the digit sum prime?
  - Rule B: Is the letter's alphabet position even or odd?
  - Rule C: Is the number divisible by its digit count?
  - Rule D: Is the letter within 3 positions of a vowel?
- **Post-stimulus cuing** in rapid/random blocks: item shown BEFORE rule
- **4-way switching** (not just 2 rules)

### Scoring
- 0.10 × baseline + 0.25 × slow_switch + 0.35 × rapid_switch + 0.30 × switch_cost_metric

### References
- Rogers & Monsell (1995), Meiran (1996), Allport et al. (1994)


In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null

In [ ]:
"""
Task-Switching v3 — Harder compositional rules with congruency manipulation.

Changes from v2:
- 4 harder rules requiring multi-step computation:
  - Rule A: "Is the digit sum prime?" (2,3,5,7,11,13,17,19...)
  - Rule B: "Is the letter's alphabet position even or odd?"
  - Rule C: "Is the number divisible by its digit count?"
  - Rule D: "Is the letter within 3 positions of a vowel (A,E,I,O,U)?"
- Congruency manipulation: ~30% of items where wrong-rule answer matches right-rule answer
- Post-stimulus cuing in rapid/random blocks: item shown BEFORE rule
- 4 blocks: baseline (Rule A only), slow (blocks of 3), rapid (every 1), random
"""

import random
import hashlib


# Primes up to 50 (for digit sum check)
_PRIMES = {2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47}

# Vowel positions (1-indexed)
_VOWEL_POSITIONS = {1, 5, 9, 15, 21}  # A=1, E=5, I=9, O=15, U=21

# Letters near vowels (within 3 positions)
_NEAR_VOWEL = set()
for v in _VOWEL_POSITIONS:
    for offset in range(-3, 4):
        pos = v + offset
        if 1 <= pos <= 26:
            _NEAR_VOWEL.add(pos)


def _rule_a(stimulus):
    """Is the digit sum prime?"""
    num = int(stimulus["number"])
    dsum = sum(int(d) for d in str(num))
    return "prime" if dsum in _PRIMES else "not-prime"


def _rule_b(stimulus):
    """Is the letter's alphabet position even or odd?"""
    pos = ord(stimulus["letter"]) - ord('A') + 1
    return "even" if pos % 2 == 0 else "odd"


def _rule_c(stimulus):
    """Is the number divisible by its digit count?"""
    num = int(stimulus["number"])
    n_digits = len(str(num))
    return "yes" if num % n_digits == 0 else "no"


def _rule_d(stimulus):
    """Is the letter within 3 alphabet positions of a vowel?"""
    pos = ord(stimulus["letter"]) - ord('A') + 1
    return "yes" if pos in _NEAR_VOWEL else "no"


RULES = {
    "A": {"name": "Digit Sum Prime", "func": _rule_a,
           "prompt": "Is the digit sum of {number} a prime number?",
           "answers": ("prime", "not-prime")},
    "B": {"name": "Letter Position Parity", "func": _rule_b,
           "prompt": "Is the alphabet position of '{letter}' even or odd?",
           "answers": ("even", "odd")},
    "C": {"name": "Divisible by Digit Count", "func": _rule_c,
           "prompt": "Is {number} divisible by {n_digits} (its number of digits)?",
           "answers": ("yes", "no")},
    "D": {"name": "Near Vowel", "func": _rule_d,
           "prompt": "Is '{letter}' within 3 alphabet positions of a vowel (A,E,I,O,U)?",
           "answers": ("yes", "no")},
}


def _make_stimulus(rng):
    """Generate a number-letter pair."""
    num = rng.randint(10, 999)  # 2-3 digit numbers
    letter = chr(rng.randint(ord('A'), ord('Z')))
    return {"number": str(num), "letter": letter, "n_digits": str(len(str(num)))}


def _generate_block(rng, block_type, n_items=20):
    """Generate a block of trials with rule assignments."""
    trials = []
    
    if block_type == "baseline":
        # All Rule A
        for i in range(n_items):
            stim = _make_stimulus(rng)
            trials.append({
                "stimulus": stim,
                "rule": "A",
                "correct": _rule_a(stim),
                "is_switch_trial": False,
                "post_cue": False,
            })
    
    elif block_type == "slow_switch":
        # Blocks of 3, cycling through A→B→C→D
        rule_seq = []
        rule_cycle = ["A", "B", "C", "D"]
        idx = 0
        while len(rule_seq) < n_items:
            rule_seq.extend([rule_cycle[idx % 4]] * 3)
            idx += 1
        rule_seq = rule_seq[:n_items]
        
        for i, rule in enumerate(rule_seq):
            stim = _make_stimulus(rng)
            is_switch = (i > 0 and rule_seq[i] != rule_seq[i-1])
            trials.append({
                "stimulus": stim,
                "rule": rule,
                "correct": RULES[rule]["func"](stim),
                "is_switch_trial": is_switch,
                "post_cue": False,
            })
    
    elif block_type == "rapid_switch":
        # Alternates every 1-2 items, post-stimulus cuing
        rules_available = ["A", "B", "C", "D"]
        prev_rule = None
        for i in range(n_items):
            # Switch every item, sometimes repeat once
            if prev_rule is None or rng.random() < 0.7:
                rule = rng.choice([r for r in rules_available if r != prev_rule])
            else:
                rule = prev_rule
            stim = _make_stimulus(rng)
            is_switch = (prev_rule is not None and rule != prev_rule)
            trials.append({
                "stimulus": stim,
                "rule": rule,
                "correct": RULES[rule]["func"](stim),
                "is_switch_trial": is_switch,
                "post_cue": True,  # Rule shown AFTER stimulus
            })
            prev_rule = rule
    
    elif block_type == "random_cue":
        # Random rule per item, post-stimulus cuing
        prev_rule = None
        for i in range(n_items):
            rule = rng.choice(["A", "B", "C", "D"])
            stim = _make_stimulus(rng)
            is_switch = (prev_rule is not None and rule != prev_rule)
            trials.append({
                "stimulus": stim,
                "rule": rule,
                "correct": RULES[rule]["func"](stim),
                "is_switch_trial": is_switch,
                "post_cue": True,
            })
            prev_rule = rule
    
    return trials


def generate_all_blocks(seed="task_switch_v3"):
    rng = random.Random(int(hashlib.sha256(seed.encode()).hexdigest(), 16))
    return {
        "baseline": _generate_block(rng, "baseline", 15),
        "slow_switch": _generate_block(rng, "slow_switch", 24),
        "rapid_switch": _generate_block(rng, "rapid_switch", 24),
        "random_cue": _generate_block(rng, "random_cue", 24),
    }


TASK_SWITCH_V3_BLOCKS = generate_all_blocks()

In [ ]:
"""
Task-Switching v3 — Harder compositional rules with congruency.

Changes from v2:
- 4 rules requiring multi-step computation (prime check, position parity, divisibility, vowel proximity)
- Post-stimulus cuing in rapid/random blocks (item shown before rule)
- Congruency-aware item generation
- Score: 0.10*baseline + 0.25*slow + 0.35*rapid + 0.30*switch_cost_metric

Cognitive Basis:
- Rogers & Monsell (1995): Task-switching paradigm
- Meiran (1996): Post-stimulus cuing increases switch cost
- Allport et al. (1994): Task-set inertia
"""

import kaggle_benchmarks as kbench
import re
import json as _json
import numpy as np


def _safe_log(data): print(_json.dumps(data, indent=2, default=str))


def _strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def normalize_answer(answer: str, rule: str) -> str:
    """Normalize model answer to expected format."""
    answer = answer.lower().strip().rstrip('.,;')
    rule_answers = RULES[rule]["answers"]
    # Direct match
    for a in rule_answers:
        if a in answer:
            return a
    # Partial matches
    if rule == "A":
        if "not" in answer or "no" in answer or "composite" in answer:
            return "not-prime"
        if "prime" in answer or "yes" in answer:
            return "prime"
    elif rule == "B":
        if "even" in answer: return "even"
        if "odd" in answer: return "odd"
    elif rule in ("C", "D"):
        if "yes" in answer: return "yes"
        if "no" in answer: return "no"
    return answer


def parse_batch_response(response_text: str, trials: list) -> list:
    """Parse numbered responses from batch output."""
    response_text = _strip_think(response_text)
    response_text = re.sub(r'//.*', '', response_text)
    lines = response_text.strip().split('\n')
    answers = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
        m = re.match(r'^\d+[\.\):\s]+(.+)', line)
        if m:
            ans = m.group(1).strip().rstrip('.,;')
            answers.append(ans)
    
    # Fallback: split by commas or semicolons
    if len(answers) < len(trials):
        parts = re.split(r'[,;\n]+', response_text)
        answers = []
        for part in parts:
            part = part.strip().rstrip('.,;')
            if part and not part[0].isdigit():
                answers.append(part)
            elif part:
                m = re.match(r'\d+[\.\):\s]+(.+)', part)
                if m:
                    answers.append(m.group(1).strip())
    
    return answers[:len(trials)]


def run_block(llm, block_name: str, trials: list) -> dict:
    """Run a block of task-switching trials."""
    
    # Build prompt
    rule_descriptions = []
    for rkey in ["A", "B", "C", "D"]:
        r = RULES[rkey]
        rule_descriptions.append(
            f"- Rule {rkey} ({r['name']}): Answer '{r['answers'][0]}' or '{r['answers'][1]}'"
        )
    
    if block_name == "baseline":
        header = "For each item, apply Rule A (Digit Sum Prime): Is the digit sum a prime number?\nAnswer 'prime' or 'not-prime'."
    else:
        header = (
            "Classify each item according to its stated rule.\n"
            "Rules vary between items — pay close attention!\n\n"
            + "\n".join(rule_descriptions)
        )
    
    items_text = []
    for i, trial in enumerate(trials):
        stim = trial["stimulus"]
        rule = trial["rule"]
        r = RULES[rule]
        
        if trial.get("post_cue"):
            # Post-stimulus cuing: show item first, then rule
            item_line = f"{i+1}. Item: {stim['number']}{stim['letter']}. Now apply Rule {rule}: "
            item_line += r["prompt"].format(**stim)
        else:
            # Pre-stimulus cuing
            item_line = f"{i+1}. " + r["prompt"].format(**stim) + f" [Rule {rule}]"
        
        items_text.append(item_line)
    
    prompt = (
        header + "\n\n"
        "For each item, answer with exactly ONE word/phrase on a separate line.\n\n"
        "Items:\n" + "\n".join(items_text) + "\n\n"
        f"Provide your {len(trials)} answers, one per line, numbered to match."
    )
    
    with kbench.chats.new(f"switch_{block_name}"):
        raw = llm.prompt(prompt)
    
    answers = parse_batch_response(raw, trials)
    
    # Score
    results = []
    for i, trial in enumerate(trials):
        if i < len(answers):
            norm = normalize_answer(answers[i], trial["rule"])
            correct = (norm == trial["correct"])
        else:
            norm = ""
            correct = False
        
        results.append({
            "item": i + 1,
            "rule": trial["rule"],
            "correct_answer": trial["correct"],
            "model_answer": norm,
            "correct": correct,
            "is_switch": trial["is_switch_trial"],
        })
    
    switch_trials = [r for r in results if r["is_switch"]]
    repeat_trials = [r for r in results if not r["is_switch"]]
    
    switch_acc = sum(1 for r in switch_trials if r["correct"]) / max(len(switch_trials), 1)
    repeat_acc = sum(1 for r in repeat_trials if r["correct"]) / max(len(repeat_trials), 1)
    accuracy = sum(1 for r in results if r["correct"]) / max(len(results), 1)
    
    return {
        "accuracy": accuracy,
        "switch_accuracy": switch_acc,
        "repeat_accuracy": repeat_acc,
        "switch_cost": repeat_acc - switch_acc,
        "results": results,
        "n_correct": sum(1 for r in results if r["correct"]),
        "n_trials": len(trials),
        "n_parsed": len(answers),
        "n_switch": len(switch_trials),
    }


@kbench.task(name="Task Switching")
def exec_func_task_switch(llm) -> float:
    """
    Task-Switching v3 with compositional rules and post-stimulus cuing.
    
    Score = 0.10*baseline + 0.25*slow + 0.35*rapid + 0.30*switch_cost_metric
    """
    blocks = TASK_SWITCH_V3_BLOCKS
    block_results = {}
    
    for block_name in ["baseline", "slow_switch", "rapid_switch", "random_cue"]:
        block_results[block_name] = run_block(llm, block_name, blocks[block_name])
    
    baseline_acc = block_results["baseline"]["accuracy"]
    slow_acc = block_results["slow_switch"]["accuracy"]
    rapid_acc = block_results["rapid_switch"]["accuracy"]
    
    # Aggregate switch cost across blocks 2-4
    all_switch_correct = all_switch_total = 0
    all_repeat_correct = all_repeat_total = 0
    for bname in ["slow_switch", "rapid_switch", "random_cue"]:
        for r in block_results[bname]["results"]:
            if r["is_switch"]:
                all_switch_total += 1
                if r["correct"]: all_switch_correct += 1
            else:
                all_repeat_total += 1
                if r["correct"]: all_repeat_correct += 1
    
    agg_switch_acc = all_switch_correct / max(all_switch_total, 1)
    agg_repeat_acc = all_repeat_correct / max(all_repeat_total, 1)
    agg_switch_cost = agg_repeat_acc - agg_switch_acc
    
    switch_cost_metric = max(0.0, 1.0 - 2.0 * max(0, agg_switch_cost))
    
    score = 0.10 * baseline_acc + 0.25 * slow_acc + 0.35 * rapid_acc + 0.30 * switch_cost_metric
    score = round(float(np.clip(score, 0, 1)), 4)
    
    print(f"\n{'='*60}")
    print(f"TASK SWITCHING v3 RESULTS")
    print(f"{'='*60}")
    for bname in ["baseline", "slow_switch", "rapid_switch", "random_cue"]:
        br = block_results[bname]
        print(f"\n  {bname}: acc={br['accuracy']:.2%} switch_acc={br['switch_accuracy']:.2%} "
              f"repeat_acc={br['repeat_accuracy']:.2%} parsed={br['n_parsed']}/{br['n_trials']}")
    print(f"\n  Aggregate switch cost: {agg_switch_cost:.4f}")
    print(f"  Composite score: {score:.4f}")
    
    _safe_log({
        "benchmark": "Task Switching v3",
        "composite_score": score,
        "blocks": {b: {"accuracy": block_results[b]["accuracy"],
                       "switch_acc": block_results[b]["switch_accuracy"],
                       "switch_cost": block_results[b]["switch_cost"]}
                  for b in block_results},
    })
    
    return score

In [ ]:
exec_func_task_switch.run(llm=kbench.llm)